# LeetCode 1854 — Maximum Population Year bằng Python

Link đề: https://leetcode.com/problems/maximum-population-year/

Độ khó: **Easy** · Chủ đề: Array, Counting, Prefix Sum

Skeleton chữ ký LeetCode:

```python
class Solution:
    def maximumPopulation(self, logs: List[List[int]]) -> int:

```

**Mục tiêu học**

1. Đọc đúng khoảng **nửa mở** `[birth, death)`: người chết năm `death` **không** được đếm năm đó.
2. Ôn list 2 chiều, unpack `for birth, death in logs`, `range(a, b)`, mảng đếm với **offset** (dịch chỉ số).
3. Đi từ đếm trực tiếp từng năm `O(Y·n)` → mảng đếm `O(n·L)` → **mảng hiệu (difference array) + prefix sum** `O(n + Y)` → **sweep line** với sự kiện sắp xếp.
4. Xử lý **hòa**: lấy năm **sớm nhất** bằng phép so sánh `>` (không phải `>=`).
5. Trace, chứng minh, Big-O và 6 bài tập về đếm theo khoảng.

## 1. Bài toán yêu cầu gì?

`logs[i] = [birth_i, death_i]` là năm sinh và năm mất của người thứ `i`.

**Dân số năm `x`** = số người còn sống trong năm `x`. Người `i` được tính ở năm `x` nếu `birth_i <= x <= death_i - 1` — tức **không** tính năm họ mất.

Trả về **năm sớm nhất** có dân số **lớn nhất**.

| logs | Kết quả | Giải thích |
|---|---:|---|
| `[[1993,1999],[2000,2010]]` | 1993 | dân số max = 1, năm sớm nhất đạt 1 là 1993 |
| `[[1950,1961],[1960,1971],[1970,1981]]` | 1960 | max = 2 ở 1960 và 1970, lấy 1960 |

**Constraints:** `1 <= len(logs) <= 100`, `1950 <= birth_i < death_i <= 2050`.

Do đó mọi năm có người sống đều nằm trong `1950..2049`.

In [1]:
from typing import List

examples = [
    ([[1993, 1999], [2000, 2010]], 1993),
    ([[1950, 1961], [1960, 1971], [1970, 1981]], 1960),
]
for logs, expected in examples:
    print(logs, "->", expected)

[[1993, 1999], [2000, 2010]] -> 1993
[[1950, 1961], [1960, 1971], [1970, 1981]] -> 1960


## 2. Nền tảng Python

### 2.1. List hai chiều và unpack

`logs` là **list các list**. `logs[0]` là `[1993, 1999]`, `logs[0][1]` là `1999`. Vòng `for birth, death in logs` **mở gói** mỗi cặp thành hai biến.

In [2]:
logs = [[1993, 1999], [2000, 2010]]
print("logs[0] =", logs[0], "| logs[0][1] =", logs[0][1])
for birth, death in logs:
    print("sinh", birth, "mất", death, "-> sống", death - birth, "năm được tính")

logs[0] = [1993, 1999] | logs[0][1] = 1999
sinh 1993 mất 1999 -> sống 6 năm được tính
sinh 2000 mất 2010 -> sống 10 năm được tính


### 2.2. `range(birth, death)` — đúng khoảng nửa mở

`range(a, b)` sinh `a, a+1, ..., b-1` — **không gồm `b`**. Thật trùng hợp, đó đúng là các năm người đó được tính: `birth .. death-1`.

In [3]:
birth, death = 1960, 1963
print(list(range(birth, death)), "<- không có 1963 (năm mất)")

[1960, 1961, 1962] <- không có 1963 (năm mất)


### 2.3. Mảng đếm với offset

Năm từ 1950 tới 2050 — ta không cần list dài 2051. Dùng list 101 ô và **dịch chỉ số**: ô `year - 1950` lưu dân số năm `year`.

In [4]:
BASE = 1950
pop = [0] * 101            # năm 1950..2050
year = 1993
pop[year - BASE] += 1
print("ô của 1993 là", year - BASE, "-> pop[43] =", pop[43])
print("ô 0 ứng với năm", 0 + BASE, ", ô 100 ứng với năm", 100 + BASE)

ô của 1993 là 43 -> pop[43] = 1
ô 0 ứng với năm 1950 , ô 100 ứng với năm 2050


### 2.4. Chọn năm sớm nhất khi hòa

Duyệt năm **tăng dần** và chỉ cập nhật khi **lớn hơn hẳn** (`>`). Khi hòa (`==`) giữ năm cũ — vốn sớm hơn.

In [5]:
counts = {1960: 2, 1965: 1, 1970: 2}
best_year, best = None, -1
for y in sorted(counts):
    if counts[y] > best:          # dùng > để giữ năm sớm khi hòa
        best, best_year = counts[y], y
print("năm sớm nhất có max:", best_year, "với", best, "người")

năm sớm nhất có max: 1960 với 2 người


`sorted(dict)` trả về list **key** đã sắp xếp.

## 3. Cách 1 — đếm trực tiếp từng năm (làm theo gợi ý đề)

Với mỗi năm `x` từ 1950 đến 2050, đếm số người có `birth <= x < death`. Lấy năm sớm nhất có số đếm lớn nhất.

In [6]:
def max_pop_year_bruteforce(logs: List[List[int]]) -> int:
    best_year, best = 1950, -1
    for x in range(1950, 2051):
        alive = 0
        for birth, death in logs:
            if birth <= x < death:
                alive += 1
        if alive > best:
            best, best_year = alive, x
    return best_year

for logs, expected in examples:
    print(max_pop_year_bruteforce(logs), "| expected", expected)
    assert max_pop_year_bruteforce(logs) == expected

1993 | expected 1993
1960 | expected 1960


`birth <= x < death` là **so sánh nối** — đúng nghĩa "x thuộc [birth, death)".

### 3.1. Viết gọn bằng `sum` và `max(..., key=...)`

`sum(1 for ... if ...)` đếm số phần tử thỏa. `max(range(...), key=f)` trả phần tử có `f` lớn nhất; khi hòa, `max` trả **phần tử đầu tiên** gặp — tức năm sớm nhất vì `range` tăng dần.

In [7]:
def max_pop_year_compact(logs):
    def alive(x):
        return sum(1 for b, d in logs if b <= x < d)
    return max(range(1950, 2051), key=alive)

print(max_pop_year_compact(examples[1][0]))
print("max hòa chọn phần tử đầu:", max([3, 7, 7, 1], key=lambda v: v), max(["b", "a"], key=lambda v: 0))

1960
max hòa chọn phần tử đầu: 7 b


**Chi phí:** `Y = 101` năm × `n` người ⇒ `O(Y·n)` ≈ 10.100 phép so sánh. Với đề này hoàn toàn ổn. Nhưng nếu khoảng năm rất lớn (ví dụ `10^9` năm), cách này bất khả thi.

## 4. Cách 2 — mảng đếm: mỗi người cộng vào các năm mình sống

Đảo vòng lặp: với **mỗi người**, cộng 1 vào mọi năm `birth..death-1`. Sau đó tìm ô lớn nhất.

In [8]:
def max_pop_year_counting(logs):
    pop = [0] * 101
    for birth, death in logs:
        for y in range(birth, death):
            pop[y - 1950] += 1
    best_i = 0
    for i in range(101):
        if pop[i] > pop[best_i]:
            best_i = i
    return best_i + 1950

for logs, expected in examples:
    assert max_pop_year_counting(logs) == expected
print(max_pop_year_counting(examples[1][0]))

1960


### 4.1. Nút thắt

Mỗi người tốn `L = death - birth` bước (tối đa 100). Tổng `O(n·L)`. Vẫn nhỏ ở bài này, nhưng ý tưởng "cộng cho **từng năm** trong khoảng" sẽ chậm khi khoảng dài.

Câu hỏi: có cách nào **cộng 1 cho cả khoảng** chỉ bằng `O(1)` thao tác?

### 4.2. Insight — mảng hiệu (difference array)

Thay vì ghi dân số, ghi **sự thay đổi** dân số:

- Năm `birth`: dân số **+1** (có người sinh).
- Năm `death`: dân số **−1** (người đó không còn được tính).

Rồi **cộng dồn** (prefix sum) từ trái sang phải: giá trị cộng dồn tại năm `y` = (số người đã sinh tới `y`) − (số người đã mất tới `y`) = **dân số năm `y`**.

Mỗi người chỉ tốn **2** thao tác, bất kể sống bao lâu.

In [9]:
# Minh họa trên trục năm nhỏ
delta = {}
for b, d in [[1950, 1961], [1960, 1971], [1970, 1981]]:
    delta[b] = delta.get(b, 0) + 1
    delta[d] = delta.get(d, 0) - 1
print("delta (năm: thay đổi):", dict(sorted(delta.items())))
run = 0
for y in sorted(delta):
    run += delta[y]
    print(f"từ năm {y}: dân số = {run}")

delta (năm: thay đổi): {1950: 1, 1960: 1, 1961: -1, 1970: 1, 1971: -1, 1981: -1}
từ năm 1950: dân số = 1
từ năm 1960: dân số = 2
từ năm 1961: dân số = 1
từ năm 1970: dân số = 2
từ năm 1971: dân số = 1
từ năm 1981: dân số = 0


`dict.items()` cho cặp `(key, value)`; `sorted(...)` sắp theo key; `dict.get(k, 0)` trả 0 nếu chưa có key.

Quan sát: năm 1960 có `+1` (người 2 sinh); năm 1961 có `−1` (người 1 mất). Dân số đạt 2 ở **1960** — đúng đáp án.

### 4.3. Prefix sum là gì?

Prefix sum của `[a0, a1, a2, ...]` là `[a0, a0+a1, a0+a1+a2, ...]`. `itertools.accumulate` tính sẵn.

In [10]:
from itertools import accumulate
print(list(accumulate([1, 0, -1, 2, -2])))

[1, 1, 0, 2, 0]


## 5. Cách 3 — lời giải chính: difference array + prefix sum

**Đoạn 1:** `delta = [0] * 101`.

**Đoạn 2:** với mỗi người: `delta[birth - 1950] += 1`, `delta[death - 1950] -= 1`. (`death` có thể là 2050 ⇒ ô 100 — vì vậy list cần 101 ô.)

**Đoạn 3:** duyệt `i = 0..100`, cộng dồn `cur += delta[i]`; nếu `cur > best` thì cập nhật `best` và `best_year = 1950 + i`.

In [11]:
def max_pop_year_diff(logs: List[List[int]]) -> int:
    delta = [0] * 101
    for birth, death in logs:
        delta[birth - 1950] += 1
        delta[death - 1950] -= 1
    cur = best = 0
    best_year = 1950
    for i in range(101):
        cur += delta[i]
        if cur > best:
            best = cur
            best_year = 1950 + i
    return best_year

for logs, expected in examples:
    assert max_pop_year_diff(logs) == expected
print(max_pop_year_diff(examples[0][0]), max_pop_year_diff(examples[1][0]))

1993 1960


Ghi chú: khởi tạo `best = 0` là an toàn vì luôn có ít nhất 1 người ⇒ dân số max ≥ 1 > 0, nên `best_year` chắc chắn được cập nhật đúng.

## 6. TRACE từng bước

`logs = [[1950,1961],[1960,1971],[1970,1981]]`. Chỉ in các năm có `delta ≠ 0`:

| năm | delta | cur (sau cộng) | best (trước) | cập nhật? |
|---:|---:|---:|---:|---|
| 1950 | +1 | 1 | 0 | có ⇒ best=1, năm 1950 |
| 1960 | +1 | 2 | 1 | có ⇒ best=2, năm 1960 |
| 1961 | −1 | 1 | 2 | không |
| 1970 | +1 | 2 | 2 | **không** (hòa, giữ 1960) |
| 1971 | −1 | 1 | 2 | không |
| 1981 | −1 | 0 | 2 | không |

In [12]:
def trace_diff(logs):
    delta = [0] * 101
    for birth, death in logs:
        delta[birth - 1950] += 1
        delta[death - 1950] -= 1
        print(f"người [{birth},{death}): delta[{birth - 1950}] += 1, delta[{death - 1950}] -= 1")
    cur = best = 0
    best_year = 1950
    print(" năm | delta | cur | best trước | hành động")
    for i in range(101):
        cur += delta[i]
        if delta[i] != 0:
            action = "cập nhật" if cur > best else ("hòa -> giữ năm cũ" if cur == best else "")
            print(f"{1950 + i} | {delta[i]:>+5} | {cur:>3} | {best:>10} | {action}")
        if cur > best:
            best, best_year = cur, 1950 + i
    print("=> năm", best_year, "dân số", best)
    return best_year

assert trace_diff([[1950, 1961], [1960, 1971], [1970, 1981]]) == 1960

người [1950,1961): delta[0] += 1, delta[11] -= 1
người [1960,1971): delta[10] += 1, delta[21] -= 1
người [1970,1981): delta[20] += 1, delta[31] -= 1
 năm | delta | cur | best trước | hành động
1950 |    +1 |   1 |          0 | cập nhật
1960 |    +1 |   2 |          1 | cập nhật
1961 |    -1 |   1 |          2 | 
1970 |    +1 |   2 |          2 | hòa -> giữ năm cũ
1971 |    -1 |   1 |          2 | 
1981 |    -1 |   0 |          2 | 
=> năm 1960 dân số 2


`{x:>+5}` in số có **dấu** (`+1`, `-1`), căn phải độ rộng 5.

Trace thứ hai: một người mất đúng năm người khác sinh — ranh giới nửa mở khiến hai người **không** cùng được đếm.

In [13]:
assert trace_diff([[1990, 2000], [2000, 2010], [1995, 2005]]) == 1995

người [1990,2000): delta[40] += 1, delta[50] -= 1
người [2000,2010): delta[50] += 1, delta[60] -= 1
người [1995,2005): delta[45] += 1, delta[55] -= 1
 năm | delta | cur | best trước | hành động
1990 |    +1 |   1 |          0 | cập nhật
1995 |    +1 |   2 |          1 | cập nhật
2005 |    -1 |   1 |          2 | 
2010 |    -1 |   0 |          2 | 
=> năm 1995 dân số 2


Năm 2000: `+1` (người 2 sinh) và `−1` (người 1 mất) **triệt tiêu** thành `delta = 0` (nên dòng năm 2000 không được in), dân số vẫn 2 — không bao giờ lên 3.

## 7. Cách 4 — sweep line với sự kiện sắp xếp

Không cần biết trước khoảng năm: tạo list **sự kiện** `(năm, +1)` khi sinh và `(năm, −1)` khi mất, **sắp xếp**, rồi quét.

Mấu chốt khi cùng năm: phải xử lý `−1` **trước** `+1` (người mất năm đó không còn tính). Sắp xếp tuple `(năm, thay_đổi)` tự làm điều này vì `−1 < +1`.

In [14]:
def max_pop_year_sweep(logs):
    events = []
    for birth, death in logs:
        events.append((birth, 1))
        events.append((death, -1))
    events.sort()                 # theo năm, cùng năm thì -1 trước +1
    cur = best = 0
    best_year = None
    for year, change in events:
        cur += change
        if cur > best:
            best, best_year = cur, year
    return best_year

print(sorted([(2000, 1), (2000, -1), (1990, 1)]))
for logs, expected in examples:
    assert max_pop_year_sweep(logs) == expected
print(max_pop_year_sweep([[1990, 2000], [2000, 2010], [1995, 2005]]))

[(1990, 1), (2000, -1), (2000, 1)]
1995


Tuple được so sánh **lần lượt từng phần tử**: `(2000, -1) < (2000, 1)` vì phần tử đầu bằng nhau và `-1 < 1`.

Vì sao `cur > best` chỉ xảy ra sau một `+1`? Dân số chỉ **tăng** khi có người sinh, nên năm cập nhật luôn là một năm sinh — chính là năm sớm nhất đạt mức đó.

## 8. Code nộp LeetCode

### Cách A — difference array (khuyên dùng)

In [15]:
class Solution:
    def maximumPopulation(self, logs: List[List[int]]) -> int:
        delta = [0] * 101
        for birth, death in logs:
            delta[birth - 1950] += 1
            delta[death - 1950] -= 1
        cur = best = 0
        best_year = 1950
        for i in range(101):
            cur += delta[i]
            if cur > best:
                best, best_year = cur, 1950 + i
        return best_year

print(Solution().maximumPopulation(examples[1][0]))

1960


### Cách B — đếm trực tiếp (ngắn, dễ kiểm)

In [16]:
class SolutionBrute:
    def maximumPopulation(self, logs: List[List[int]]) -> int:
        return max(range(1950, 2051), key=lambda x: sum(b <= x < d for b, d in logs))

print(SolutionBrute().maximumPopulation(examples[1][0]))

1960


`sum(b <= x < d for ...)` cộng các giá trị `True/False` — `True` được tính là 1.

### Cách C — sweep line

In [17]:
class SolutionSweep:
    def maximumPopulation(self, logs: List[List[int]]) -> int:
        events = sorted([(b, 1) for b, _ in logs] + [(d, -1) for _, d in logs])
        cur = best = 0
        ans = 0
        for year, ch in events:
            cur += ch
            if cur > best:
                best, ans = cur, year
        return ans

print(SolutionSweep().maximumPopulation(examples[1][0]))

1960


Dấu `_` là tên biến quy ước cho giá trị **không dùng tới**.

## 9. Vì sao đúng?

**Difference array:** Với mỗi người, `delta[birth] += 1` và `delta[death] -= 1`. Sau cộng dồn, đóng góp của người đó vào `cur` tại năm `y` là:

- `0` nếu `y < birth` (chưa gặp `+1`),
- `1` nếu `birth <= y < death` (đã gặp `+1`, chưa gặp `−1`),
- `0` nếu `y >= death` (gặp cả hai, triệt tiêu).

Đúng bằng định nghĩa "được tính ở năm `y`". Cộng mọi người ⇒ `cur` = dân số năm `y`.

**Invariant khi quét:** sau khi xử lý ô `i`, `best` là dân số lớn nhất của các năm `1950..1950+i`, và `best_year` là năm **sớm nhất** đạt `best` (vì chỉ cập nhật khi tăng **hẳn**).

In [18]:
# Kiểm chứng: prefix sum của delta trùng với mảng đếm trực tiếp ở MỌI năm
import random
rng = random.Random(1854)
for _ in range(300):
    logs = []
    for _ in range(rng.randint(1, 15)):
        b = rng.randint(1950, 2049)
        logs.append([b, rng.randint(b + 1, 2050)])
    delta = [0] * 101
    for b, d in logs:
        delta[b - 1950] += 1
        delta[d - 1950] -= 1
    direct = [sum(b <= 1950 + i < d for b, d in logs) for i in range(101)]
    assert list(accumulate(delta)) == direct
print("prefix sum của delta == dân số trực tiếp ở mọi năm (300 bộ)")

prefix sum của delta == dân số trực tiếp ở mọi năm (300 bộ)


## 10. Độ phức tạp

Gọi `n = len(logs)`, `Y = 101` (số năm), `L` = tuổi thọ tối đa (≤ 100).

| Cách | Time | Space |
|---|---|---|
| 1. Đếm từng năm | `O(Y·n)` | `O(1)` |
| 2. Mảng đếm | `O(n·L + Y)` | `O(Y)` |
| 3. Difference array | `O(n + Y)` | `O(Y)` |
| 4. Sweep line | `O(n log n)` | `O(n)` |

Cách 3 tốt nhất khi khoảng năm nhỏ & cố định; cách 4 tốt khi khoảng năm **rất lớn** nhưng số người ít (không phụ thuộc `Y`).

## 11. Bộ test

Nhóm: mẫu đề; một người; mọi người trùng khoảng; người mất đúng năm người khác sinh; hòa nhiều năm; biên 1950/2050; ngẫu nhiên đối chiếu 7 cài đặt.

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS ({actual})")

sol = Solution()
check("mẫu 1", sol.maximumPopulation([[1993, 1999], [2000, 2010]]), 1993)
check("mẫu 2", sol.maximumPopulation([[1950, 1961], [1960, 1971], [1970, 1981]]), 1960)
check("một người", sol.maximumPopulation([[2049, 2050]]), 2049)
check("trùng khoảng", sol.maximumPopulation([[1980, 1990]] * 3), 1980)
check("mất = sinh của người khác", sol.maximumPopulation([[1990, 2000], [2000, 2010]]), 1990)
check("biên 1950..2050", sol.maximumPopulation([[1950, 2050], [2049, 2050]]), 2049)
check("lồng nhau", sol.maximumPopulation([[1950, 2000], [1960, 1990], [1970, 1980]]), 1970)

mẫu 1                        PASS (1993)
mẫu 2                        PASS (1960)
một người                    PASS (2049)
trùng khoảng                 PASS (1980)
mất = sinh của người khác    PASS (1990)
biên 1950..2050              PASS (2049)
lồng nhau                    PASS (1970)


In [20]:
impls = [max_pop_year_bruteforce, max_pop_year_compact, max_pop_year_counting, max_pop_year_diff,
         max_pop_year_sweep, Solution().maximumPopulation, SolutionBrute().maximumPopulation,
         SolutionSweep().maximumPopulation]
for _ in range(500):
    logs = []
    for _ in range(rng.randint(1, 30)):
        b = rng.randint(1950, 2049)
        logs.append([b, rng.randint(b + 1, min(2050, b + rng.choice([1, 5, 30, 100])))])
    res = {f(logs) for f in impls}
    assert len(res) == 1, (logs, res)
print("8 cài đặt đồng ý trên 500 bộ ngẫu nhiên PASS")

8 cài đặt đồng ý trên 500 bộ ngẫu nhiên PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: tính cả năm mất (khoảng đóng)

In [21]:
def wrong_inclusive_death(logs):
    best_year, best = 1950, -1
    for x in range(1950, 2051):
        alive = sum(b <= x <= d for b, d in logs)   # <= d là sai
        if alive > best:
            best, best_year = alive, x
    return best_year

logs = [[1990, 2000], [2000, 2010]]
print("sai :", wrong_inclusive_death(logs), "| đúng:", max_pop_year_diff(logs))

sai : 2000 | đúng: 1990


Năm 2000 bị đếm 2 người. Sửa: `b <= x < d` (hoặc `delta[death] -= 1`, **không** phải `delta[death + 1]`).

### Lỗi 2: dùng `>=` khi cập nhật ⇒ lấy năm muộn nhất

In [22]:
def wrong_ge(logs):
    delta = [0] * 101
    for b, d in logs:
        delta[b - 1950] += 1
        delta[d - 1950] -= 1
    cur = best = 0
    year = 1950
    for i in range(101):
        cur += delta[i]
        if cur >= best:
            best, year = cur, 1950 + i
    return year

logs = examples[1][0]
print("sai :", wrong_ge(logs), "| đúng:", max_pop_year_diff(logs))

sai : 1970 | đúng: 1960


Với `>=`, mỗi lần **hòa** kỷ lục đều ghi đè năm cũ: năm 1970 (dân số 2, bằng kỷ lục) thay thế 1960. Kết quả là năm **muộn nhất** đạt max. Sửa: `>`.

### Lỗi 3: mảng chỉ 100 ô ⇒ `death = 2050` tràn chỉ số

In [23]:
def wrong_size(logs):
    delta = [0] * 100
    for b, d in logs:
        delta[b - 1950] += 1
        delta[d - 1950] -= 1
    return delta

try:
    wrong_size([[2000, 2050]])
except IndexError as e:
    print("Lỗi:", e, "-> cần 101 ô (1950..2050)")

Lỗi: list index out of range -> cần 101 ô (1950..2050)


### Lỗi 4: sweep line sắp xếp `+1` trước `−1` cùng năm

In [24]:
def wrong_sweep_order(logs):
    events = []
    for b, d in logs:
        events.append((b, 0))     # mã 0 = sinh (đứng trước)
        events.append((d, 1))     # mã 1 = mất (đứng sau) -> sai thứ tự
    events.sort()
    cur = best = 0
    ans = None
    for year, kind in events:
        cur += 1 if kind == 0 else -1
        if cur > best:
            best, ans = cur, year
    return ans, best

print("sai :", wrong_sweep_order([[1990, 2000], [2000, 2010]]), "(tưởng max = 2 ở năm 2000)")
print("đúng:", max_pop_year_sweep([[1990, 2000], [2000, 2010]]), "(max = 1)")

sai : (2000, 2) (tưởng max = 2 ở năm 2000)
đúng: 1990 (max = 1)


Sửa: cùng năm thì xử lý **mất trước sinh** — dùng `(năm, -1)` và `(năm, +1)` để sort tự đúng.

## 13. Mẹo nhận dạng pattern

- Nhiều **khoảng** `[l, r)` + hỏi "điểm nào bị phủ nhiều nhất / phủ bao nhiêu lần" ⇒ **difference array** (`+1` ở `l`, `−1` ở `r`, rồi prefix sum).
- Miền tọa độ **lớn** hoặc thưa ⇒ **sweep line**: sắp xếp sự kiện.
- Luôn tự hỏi: biên phải **đóng hay mở**? Nếu đóng `[l, r]` thì `−1` đặt ở `r + 1`.
- "Sớm nhất / nhỏ nhất khi hòa" ⇒ duyệt tăng dần và so sánh `>`.
- Miền giá trị cố định nhỏ (1950..2050) ⇒ list với **offset** thay cho dict.

## 14. Biến thể

1. Trả về **dân số lớn nhất** thay vì năm.
2. **Khoảng đóng** `[birth, death]` (tính cả năm mất).
3. **LeetCode 1094 — Car Pooling**: mỗi chuyến `[người, từ, đến]`, kiểm tra không lúc nào vượt sức chứa.
4. **LeetCode 253 — Meeting Rooms II**: số phòng họp tối thiểu = số cuộc họp chồng nhau tối đa ⇒ sweep line.

In [25]:
def max_overlap(intervals):
    events = sorted([(s, 1) for s, _ in intervals] + [(e, -1) for _, e in intervals])
    cur = best = 0
    for _, ch in events:
        cur += ch
        best = max(best, cur)
    return best

print("Meeting rooms [[0,30],[5,10],[15,20]] ->", max_overlap([[0, 30], [5, 10], [15, 20]]))
print("Meeting rooms [[7,10],[2,4]] ->", max_overlap([[7, 10], [2, 4]]))

Meeting rooms [[0,30],[5,10],[15,20]] -> 2
Meeting rooms [[7,10],[2,4]] -> 1


## 15. Bài tập (tự làm trước)

**Bài 1 — tính tay:** `logs = [[2000,2005],[2003,2008],[2004,2006],[2007,2010]]`. Lập bảng delta & cur; năm trả về là gì, dân số max bao nhiêu?

**Bài 2 — dự đoán output:**

```python
print(sorted([(2000, 1), (1999, -1), (2000, -1)]))
print(list(range(2000, 2003)))
```

**Bài 3 — điền TODO:** difference array.

In [26]:
def exercise_max_pop_year(logs):
    # TODO 1: delta = [0] * 101
    # TODO 2: +1 tại birth - 1950, -1 tại death - 1950
    # TODO 3: cộng dồn, cập nhật khi cur > best
    pass

# assert exercise_max_pop_year([[1993, 1999], [2000, 2010]]) == 1993
# assert exercise_max_pop_year([[1950, 1961], [1960, 1971], [1970, 1981]]) == 1960

**Bài 4 — biến thể khoảng đóng:** người được tính cả năm mất. Trả về năm sớm nhất có dân số lớn nhất (năm có thể tới 2050).

In [27]:
def exercise_max_pop_inclusive(logs):
    # TODO: -1 đặt ở death + 1 -> cần thêm một ô
    pass

# assert exercise_max_pop_inclusive([[1990, 2000], [2000, 2010]]) == 2000
# assert exercise_max_pop_inclusive([[1993, 1999], [2000, 2010]]) == 1993

**Bài 5 — Car Pooling (LC 1094):** `trips[i] = [num, from, to]` (đón ở `from`, trả ở `to`, `0 <= from < to <= 1000`). Trả `True` nếu không lúc nào số khách vượt `capacity`.

In [28]:
def exercise_car_pooling(trips, capacity):
    # TODO: difference array 1001 ô, +num tại from, -num tại to
    pass

# assert exercise_car_pooling([[2, 1, 5], [3, 3, 7]], 4) is False
# assert exercise_car_pooling([[2, 1, 5], [3, 3, 7]], 5) is True

**Bài 6 — mini challenge:** miền năm **rất lớn** (`1 <= birth < death <= 10^9`). Viết `exercise_max_pop_big(logs)` trả năm sớm nhất có dân số max, không tạo list dài `10^9`.

In [29]:
def exercise_max_pop_big(logs):
    # TODO: sweep line với sự kiện (năm, -1)/(năm, +1)
    pass

# assert exercise_max_pop_big([[1, 10**9], [5 * 10**8, 6 * 10**8]]) == 5 * 10**8
# assert exercise_max_pop_big([[10, 20], [20, 30]]) == 10

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | ghi +1/−1 tại năm sinh/mất | sắp theo năm rồi cộng dồn | max = 3 tại 2004 |
| 2 | tuple so từng phần tử | `(2000,-1)` trước `(2000,1)` | `range` không gồm 2003 |
| 3 | 101 ô | offset 1950 | `if cur > best` |
| 4 | khoảng đóng ⇒ trừ ở `death+1` | list 102 ô | duyệt `i` tới 100 |
| 5 | giống dân số | `delta[f] += num`, `delta[t] -= num` | nếu `cur > capacity` trả False |
| 6 | không thể dùng list | events + sort | cập nhật khi `cur > best` |

## 17. Đáp án tham khảo

**Bài 1:** delta: 2000 +1, 2003 +1, 2004 +1, 2005 −1, 2006 −1, 2007 +1, 2008 −1, 2010 −1 ⇒ cur: 1, 2, **3**, 2, 1, 2, 1, 0 ⇒ năm **2004**, dân số max **3**.

**Bài 2:** `[(1999, -1), (2000, -1), (2000, 1)]` và `[2000, 2001, 2002]`.

In [30]:
print(trace_diff([[2000, 2005], [2003, 2008], [2004, 2006], [2007, 2010]]))
print(sorted([(2000, 1), (1999, -1), (2000, -1)]))
print(list(range(2000, 2003)))

người [2000,2005): delta[50] += 1, delta[55] -= 1
người [2003,2008): delta[53] += 1, delta[58] -= 1
người [2004,2006): delta[54] += 1, delta[56] -= 1
người [2007,2010): delta[57] += 1, delta[60] -= 1
 năm | delta | cur | best trước | hành động
2000 |    +1 |   1 |          0 | cập nhật
2003 |    +1 |   2 |          1 | cập nhật
2004 |    +1 |   3 |          2 | cập nhật
2005 |    -1 |   2 |          3 | 
2006 |    -1 |   1 |          3 | 
2007 |    +1 |   2 |          3 | 
2008 |    -1 |   1 |          3 | 
2010 |    -1 |   0 |          3 | 
=> năm 2004 dân số 3
2004
[(1999, -1), (2000, -1), (2000, 1)]
[2000, 2001, 2002]


In [31]:
def answer_max_pop_year(logs):
    delta = [0] * 101
    for b, d in logs:
        delta[b - 1950] += 1
        delta[d - 1950] -= 1
    cur = best = 0
    ans = 1950
    for i in range(101):
        cur += delta[i]
        if cur > best:
            best, ans = cur, 1950 + i
    return ans


def answer_max_pop_inclusive(logs):
    delta = [0] * 102
    for b, d in logs:
        delta[b - 1950] += 1
        delta[d - 1950 + 1] -= 1
    cur = best = 0
    ans = 1950
    for i in range(101):
        cur += delta[i]
        if cur > best:
            best, ans = cur, 1950 + i
    return ans


def answer_car_pooling(trips, capacity):
    delta = [0] * 1001
    for num, f, t in trips:
        delta[f] += num
        delta[t] -= num
    cur = 0
    for v in delta:
        cur += v
        if cur > capacity:
            return False
    return True


def answer_max_pop_big(logs):
    events = sorted([(b, 1) for b, _ in logs] + [(d, -1) for _, d in logs])
    cur = best = 0
    ans = None
    for year, ch in events:
        cur += ch
        if cur > best:
            best, ans = cur, year
    return ans


assert answer_max_pop_year([[1993, 1999], [2000, 2010]]) == 1993
assert answer_max_pop_year([[1950, 1961], [1960, 1971], [1970, 1981]]) == 1960
assert answer_max_pop_inclusive([[1990, 2000], [2000, 2010]]) == 2000
assert answer_max_pop_inclusive([[1993, 1999], [2000, 2010]]) == 1993
assert answer_car_pooling([[2, 1, 5], [3, 3, 7]], 4) is False
assert answer_car_pooling([[2, 1, 5], [3, 3, 7]], 5) is True
assert answer_max_pop_big([[1, 10**9], [5 * 10**8, 6 * 10**8]]) == 5 * 10**8
assert answer_max_pop_big([[10, 20], [20, 30]]) == 10
for _ in range(200):
    logs = []
    for _ in range(rng.randint(1, 20)):
        b = rng.randint(1950, 2049)
        logs.append([b, rng.randint(b + 1, 2050)])
    assert answer_max_pop_big(logs) == answer_max_pop_year(logs) == max_pop_year_bruteforce(logs)
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def maximumPopulation(self, logs: List[List[int]]) -> int:
        delta = [0] * 101                      # năm 1950..2050
        for b, d in logs:
            delta[b - 1950] += 1               # sinh: +1
            delta[d - 1950] -= 1               # mất: -1 (năm mất không tính)
        cur = best = 0
        ans = 1950
        for i in range(101):
            cur += delta[i]                    # prefix sum = dân số năm 1950 + i
            if cur > best:                     # > để giữ năm sớm nhất
                best, ans = cur, 1950 + i
        return ans
```

Khung "difference array": **`+v` ở đầu khoảng, `−v` ngay sau cuối khoảng, cộng dồn.**

Checklist:

- [ ] Khoảng là `[birth, death)` — năm mất **không** tính.
- [ ] List đủ 101 ô (có ô cho 2050).
- [ ] Offset `- 1950` khi ghi, `+ 1950` khi trả.
- [ ] So sánh `>` để lấy năm **sớm nhất** khi hòa.
- [ ] Sweep line: cùng năm thì `−1` trước `+1`.